In [ ]:
!pip install -q transformers pandas scikit-learn
import os
import json
import re
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/CS4248/"
ANNOTATIONS_FILE = os.path.join(DATA_PATH, "lstm/training_data.csv")
TEXT_COL = "clean_text" 
LABEL_COL = "sentiment"
LABEL_NAMES = ["negative", "neutral", "positive"]

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_LEN = 64

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps') # M1/M2/M3 Mac GPU support
else:
    DEVICE = torch.device('cpu')

In [ ]:
# Toggle between 'subword' (HuggingFace AutoTokenizer) and 'word' (Custom Classic Vocab)
TOKENIZER_TYPE = 'word'  # Set to 'subword' for RoBERTa comparison, 'word' for classic baseline

class CustomWordTokenizer:
    """A classic word-level tokenizer that builds a vocabulary based on word frequency."""
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def fit(self, texts):
        counter = Counter()
        for text in texts:
            words = str(text).lower().split()
            counter.update(words)
            
        for word, _ in counter.most_common(self.max_vocab_size - 2):
            self.word2idx[word] = self.vocab_size
            self.idx2word[self.vocab_size] = word
            self.vocab_size += 1
            
    def encode_plus(self, text, max_length, **kwargs):
        words = str(text).lower().split()
        input_ids = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in words]
        
        # pad or truncate
        if len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
        else:
            input_ids = input_ids + [self.word2idx['<PAD>']] * (max_length - len(input_ids))
            
        return {
            'input_ids': torch.tensor(input_ids).unsqueeze(0)       # unsqueeze to match HF return shape 
        }

# --- DATASET COMPONENT ---
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_map = {name: i for i, name in enumerate(LABEL_NAMES)}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = int(self.labels[item] if pd.notnull(self.labels[item]) else 1)

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

# --- MODEL DEFINITION ---
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=3,
                 num_layers=2, bidirectional=True, dropout=0.3,
                 pretrained_embeddings=None, freeze_embeddings=False):
        super(LSTMClassifier, self).__init__()
        if pretrained_embeddings is not None:
            embedding_dim = pretrained_embeddings.shape[1]
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                            bidirectional=bidirectional, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, num_classes)
        
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        if self.lstm.bidirectional:
            hidden_final = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden_final = hidden[-1,:,:]
        out = self.dropout(hidden_final)
        out = self.fc(out)
        return out

# --- TRAINING LOOP ---
def train_epoch(model, data_loader, loss_fn, optimizer, device, clip_grad=1.0):
    model = model.train()
    losses = []
    correct_predictions = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        targets = batch['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)

        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        if clip_grad:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            targets = batch['targets'].to(device)

            outputs = model(input_ids)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)

            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    avg_acc = correct_predictions.double() / len(data_loader.dataset)
    return avg_acc, np.mean(losses), all_targets, all_preds

# --- GLOVE LOADER ---
def load_glove_embeddings(glove_path, word2idx, embedding_dim):
    """Load pretrained GloVe vectors. Missing words get small random vectors instead of zeros."""
    # Random init for all words first (PAD stays zero)
    embeddings = np.random.uniform(-0.05, 0.05, (len(word2idx), embedding_dim))
    embeddings[0] = 0  # <PAD> stays zero

    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in word2idx:
                embeddings[word2idx[word]] = np.array(values[1:], dtype='float32')
                found += 1
    print(f"GloVe: {found}/{len(word2idx)} vocab words found ({found/len(word2idx)*100:.1f}%)")
    return torch.FloatTensor(embeddings)

In [ ]:
print(f"Using device: {DEVICE}")
if not os.path.exists(ANNOTATIONS_FILE):
    print(f"Error: {ANNOTATIONS_FILE} not found. Please check your DATA_PATH/Google Drive mount.")
    
print("Loading dataset...")
df = pd.read_csv(ANNOTATIONS_FILE)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[LABEL_COL])
print(f"Train size: {len(df_train)}, Test size: {len(df_test)}")

if TOKENIZER_TYPE == 'subword':
    print("\n--- Initializing Hugging Face AutoTokenizer (Subword) ---")
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    vocab_size = tokenizer.vocab_size
else:
    print("\n--- Initializing Custom Classic Vocab Tokenizer (Word-level) ---")
    tokenizer = CustomWordTokenizer(max_vocab_size=20000)
    tokenizer.fit(df_train[TEXT_COL].values)
    vocab_size = tokenizer.vocab_size

train_dataset = TweetDataset(df_train[TEXT_COL].values, df_train[LABEL_COL].values, tokenizer, MAX_LEN)
test_dataset = TweetDataset(df_test[TEXT_COL].values, df_test[LABEL_COL].values, tokenizer, MAX_LEN)

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Vocab size: {vocab_size}")

Using device: mps
Loading dataset...
Train size: 838857, Test size: 209715

--- Initializing Custom Classic Vocab Tokenizer (Word-level) ---
Vocab size: 20000


In [ ]:
model = LSTMClassifier(vocab_size=vocab_size, embedding_dim=128, hidden_dim=256, num_classes=3)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)

best_acc = 0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 10)
    
    train_acc, train_loss = train_epoch(model, train_data_loader, loss_fn, optimizer, DEVICE)
    print(f"Train loss {train_loss:.4f} accuracy {train_acc:.4f}")

    val_acc, val_loss, targets, preds = eval_model(model, test_data_loader, loss_fn, DEVICE)
    print(f"Val   loss {val_loss:.4f} accuracy {val_acc:.4f}")
    print()

    if val_acc > best_acc:
        best_acc = val_acc
        
# Final evaluation
print(f"Final Evaluation on Test Set ({TOKENIZER_TYPE} tokenization):")
_, _, final_targets, final_preds = eval_model(model, test_data_loader, loss_fn, DEVICE)
print(classification_report(final_targets, final_preds, target_names=LABEL_NAMES))

Epoch 1/5
----------


In [ ]:
# --- TRAIN ON FULL TRAINING SET, EVALUATE ON tweets.csv ---
# Uses GloVe Twitter embeddings (200d) for better domain generalisation

TWEETS_FILE = os.path.join(DATA_PATH, "lstm/tweets.csv")
TWEETS_TEXT_COL = "clean_txt"
TWEETS_LABEL_COL = "label"
GLOVE_DIM = 200  # 25 / 50 / 100 / 200 available in glove.twitter.27B
EPOCHS = 10

# Download GloVe Twitter embeddings to Drive (only runs once across sessions)
import zipfile
GLOVE_DIR = os.path.join(DATA_PATH, "glove")
GLOVE_FILE = os.path.join(GLOVE_DIR, f"glove.twitter.27B.{GLOVE_DIM}d.txt")
os.makedirs(GLOVE_DIR, exist_ok=True)
if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe Twitter embeddings to Drive (~1.4 GB)...")
    GLOVE_ZIP = os.path.join(GLOVE_DIR, "glove.twitter.27B.zip")
    os.system(f"wget -q https://nlp.stanford.edu/data/glove.twitter.27B.zip -O {GLOVE_ZIP}")
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract(f"glove.twitter.27B.{GLOVE_DIM}d.txt", GLOVE_DIR)
    os.remove(GLOVE_ZIP)  # remove zip to save Drive space
    print("Done. Saved to Drive.")
else:
    print(f"GloVe found in Drive: {GLOVE_FILE}")

# Load full training + test data
print("\nLoading datasets...")
df_full = pd.read_csv(ANNOTATIONS_FILE)
df_full = df_full.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df_tweets = pd.read_csv(TWEETS_FILE)
df_tweets = df_tweets.dropna(subset=[TWEETS_TEXT_COL, TWEETS_LABEL_COL]).copy()
print(f"Full training size: {len(df_full)}  |  Tweets test size: {len(df_tweets)}")

# Fit tokenizer on full training set
full_tokenizer = CustomWordTokenizer(max_vocab_size=20000)
full_tokenizer.fit(df_full[TEXT_COL].values)
print(f"Vocab size: {full_tokenizer.vocab_size}")

# Load pretrained GloVe embeddings aligned to vocab
glove_embeddings = load_glove_embeddings(GLOVE_FILE, full_tokenizer.word2idx, GLOVE_DIM)

# Datasets and loaders
full_train_dataset = TweetDataset(df_full[TEXT_COL].values, df_full[LABEL_COL].values, full_tokenizer, MAX_LEN)
tweets_test_dataset = TweetDataset(df_tweets[TWEETS_TEXT_COL].values, df_tweets[TWEETS_LABEL_COL].values, full_tokenizer, MAX_LEN)
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
tweets_test_loader = DataLoader(tweets_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Class weights: inverse frequency to counter neutral bias in training data
label_counts = df_full[LABEL_COL].value_counts().sort_index()  # [neg, neu, pos]
class_weights = torch.tensor(
    [label_counts.max() / label_counts[i] for i in range(3)], dtype=torch.float
).to(DEVICE)
print(f"Class weights: negative={class_weights[0]:.2f}, neutral={class_weights[1]:.2f}, positive={class_weights[2]:.2f}")

# 1-layer BiLSTM, hidden=128
full_model = LSTMClassifier(
    vocab_size=full_tokenizer.vocab_size,
    hidden_dim=128,
    num_classes=3,
    num_layers=1,
    bidirectional=True,
    dropout=0.5,
    pretrained_embeddings=glove_embeddings,
    freeze_embeddings=True,
)
full_model = full_model.to(DEVICE)

full_optimizer = torch.optim.Adam(full_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
full_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(full_optimizer, mode='min', patience=2, factor=0.5)

print(f"\nTraining on full dataset with GloVe-{GLOVE_DIM}d embeddings (1-layer BiLSTM, hidden=128, weighted loss)...")
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 10)
    train_acc, train_loss = train_epoch(full_model, full_train_loader, full_loss_fn, full_optimizer, DEVICE, clip_grad=1.0)
    print(f"Train loss {train_loss:.4f} accuracy {train_acc:.4f}")
    scheduler.step(train_loss)

# Evaluate on tweets.csv
print(f"\nFinal Evaluation on tweets.csv (GloVe-{GLOVE_DIM}d, 1-layer BiLSTM, weighted loss):")
_, _, tweets_targets, tweets_preds = eval_model(full_model, tweets_test_loader, full_loss_fn, DEVICE)
print(classification_report(tweets_targets, tweets_preds, target_names=LABEL_NAMES))
print(f"Accuracy:      {accuracy_score(tweets_targets, tweets_preds):.4f}")
print(f"Macro F1:      {f1_score(tweets_targets, tweets_preds, average='macro'):.4f}")
print(f"Cohen Kappa:   {cohen_kappa_score(tweets_targets, tweets_preds):.4f}")